# Raidium — visualisation

Reading tool for the data and the predictions. No narrative: the reasoning, the
experiments and their verdicts are in `notes.tex`.

Same shape throughout — helpers at the top of each section, then short cells that
call them. To look at something else, change an argument and re-run.

| section | what it does |
|---|---|
| 1 | load, and get the 54 organ names |
| 2 | look at one slice, or a series |
| 3 | find the slices containing a given organ |
| 4 | zoom on an organ — necessary below 200 px |
| 5 | compare a model against the truth |
| 6 | inspect the external set |
| 7 | the reference table of all 54 organs |

---
## 1. Load

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle

from radium import data, metric, totalseg

warnings.filterwarnings("ignore", message="Mean of empty slice")

X = data.train_images()      # (2000, 256, 256) uint8 — 800 annotated, then 1200 raw
Y = data.train_labels()      # (2000, 256, 256) uint8 — 0 = background *or* unoutlined organ
A = data.annotated_ids()     # (2000, 54) bool    — which organs were examined
Xt = data.test_images()      # (500, 256, 256)
ANN = np.flatnonzero(A.sum(1) == 27)   # the 800 usable slices

# All 54 names, derived and verified one for one: the challenge ids are
# TotalSegmentator v1 merged and sorted alphabetically. See radium/totalseg.py.
NAMES = totalseg.NAMES
print(f"{len(X)} slices, {len(ANN)} annotated, {len(Xt)} test")
print(f"organ 1 = {NAMES[1]}   organ 36 = {NAMES[36]}   organ 54 = {NAMES[54]}")

In [ ]:
# Fixed palette per id: the same organ keeps the same colour across figures,
# without which two panels cannot be compared by eye.
CMAP = ListedColormap(np.concatenate([
    plt.get_cmap("tab20")(np.linspace(0, 1, 20)),
    plt.get_cmap("tab20b")(np.linspace(0, 1, 20)),
    plt.get_cmap("tab20c")(np.linspace(0, 1, 20)),
])[:54])

def overlay(ax, scan, mask, highlight=None, alpha=0.55):
    """Grey scan, coloured mask, and one organ in red if asked."""
    ax.imshow(scan, cmap="gray", vmin=0, vmax=255)
    m = np.ma.masked_where(mask == 0, mask)
    ax.imshow(m, cmap=CMAP, alpha=alpha, vmin=1, vmax=54, interpolation="nearest")
    if highlight is not None:
        h = mask == highlight
        ax.imshow(np.ma.masked_where(~h, h.astype(float)), cmap="autumn",
                  alpha=0.9, interpolation="nearest")
    ax.axis("off")

def label(ids, per_line=3):
    """Ids -> names, wrapped over several lines."""
    n = [f"{c} {NAMES[c]}" for c in ids]
    return "\n".join(", ".join(n[k:k+per_line]) for k in range(0, len(n), per_line)) or "(none)"

---
## 2. Look at a slice

`show(i)` draws the bare scan, then the annotated one. The title lists the organs
actually present in the mask — not the 27 declared, which are often absent.

In [ ]:
def show(i, ax=None):
    present = sorted(np.unique(Y[i])[1:].tolist())
    if ax is None:
        _, ax = plt.subplots(1, 2, figsize=(9, 5))
    ax[0].imshow(X[i], cmap="gray", vmin=0, vmax=255); ax[0].axis("off")
    ax[0].set_title(f"slice {i}", fontsize=10)
    overlay(ax[1], X[i], Y[i])
    ax[1].set_title(f"{len(present)} organs visible of {int(A[i].sum())} examined\n"
                    + label(present), fontsize=7)
    return present

show(55)
plt.tight_layout(); plt.show()

In [ ]:
def show_many(indices, ncol=5):
    """A series of slices, scan on top, annotation below."""
    indices = list(indices)[:ncol]
    fig, ax = plt.subplots(2, len(indices), figsize=(3.1 * len(indices), 6.6), squeeze=False)
    for k, i in enumerate(indices):
        ax[0, k].imshow(X[i], cmap="gray", vmin=0, vmax=255); ax[0, k].axis("off")
        ax[0, k].set_title(f"slice {i}", fontsize=9)
        overlay(ax[1, k], X[i], Y[i])
        ax[1, k].set_title(label(sorted(np.unique(Y[i])[1:].tolist()), 2), fontsize=6)
    plt.tight_layout(); plt.show()

# The richest slices: those where the most organs are visible at once.
show_many(sorted(ANN, key=lambda i: -len(np.unique(Y[i])))[:5])

---
## 3. Find slices

Three filters covering most of the questions worth asking.

In [ ]:
def slices_with(*organs, among=None):
    """Slices where all these organs are present in the mask."""
    pool = ANN if among is None else among
    return [i for i in pool if all((Y[i] == c).any() for c in organs)]

def slices_examined(c, present=None):
    """Slices where organ c was examined. `present=False` gives the certain
    absences — the reliable negative supervision the marginal loss uses."""
    out = [i for i in ANN if A[i, c - 1]]
    if present is None:
        return out
    return [i for i in out if (Y[i] == c).any() == present]

def by_id(name):
    """Find an id from part of a name: by_id('spleen') -> 48."""
    return [c for c, n in NAMES.items() if name.lower() in n.lower()]

print("spleen:", by_id("spleen"), "| kidneys:", by_id("kidney"), "| vertebrae:", by_id("vertebrae"))
liver_and_spleen = slices_with(36, 48)
print(f"\nslices with liver AND spleen: {len(liver_and_spleen)} -> {liver_and_spleen[:8]}")
print(f"slices where the spleen was examined and is absent: {len(slices_examined(48, present=False))}")

In [ ]:
show_many(liver_and_spleen[:5])

---
## 4. Zoom on an organ

Below 200 px a full view shows nothing: the adrenals are 37 px out of 65 536.
Each row gives the whole slice with the zoom window, then magnifications in
**raw / annotated** pairs — looking at the raw one first is the only way to know
whether the information is really there.

In [ ]:
def crop(mask, side_min=72, zoom=4.0, side_max=160):
    """Square window centred on the organ; the cap keeps stretched structures readable."""
    rr, cc = np.nonzero(mask)
    r0, c0 = (rr.min() + rr.max()) // 2, (cc.min() + cc.max()) // 2
    s = int(np.clip(zoom * max(np.ptp(rr), np.ptp(cc)), side_min, side_max)) // 2
    r0, c0 = np.clip(r0, s, 256 - s), np.clip(c0, s, 256 - s)
    return slice(r0 - s, r0 + s), slice(c0 - s, c0 + s)

def pick(c, k=4):
    """k slices where the organ is largest; tangential ones show nothing."""
    idx = sorted((i for i in ANN if (Y[i] == c).any()), key=lambda i: -(Y[i] == c).sum())
    return sorted(dict.fromkeys(idx[j] for j in np.linspace(0, max(len(idx)//2 - 1, 0), k).astype(int)))

def show_organ(c, npair=4):
    picks = pick(c, npair)
    if not picks:
        print(f"organ {c} ({NAMES[c]}): absent from the annotated slices"); return
    fig, ax = plt.subplots(1, 1 + 2 * len(picks), figsize=(2.7 * (1 + 2 * len(picks)), 3.3))
    box0 = crop(Y[picks[0]] == c)
    overlay(ax[0], X[picks[0]], Y[picks[0]], highlight=c, alpha=0.35)
    ax[0].add_patch(Rectangle((box0[1].start, box0[0].start),
                              box0[1].stop - box0[1].start, box0[0].stop - box0[0].start,
                              fill=False, ec="#39FF14", lw=1.4))
    ax[0].set_title(f"slice {picks[0]} — whole", fontsize=7)
    for k, i in enumerate(picks):
        b = crop(Y[i] == c)
        ax[1 + 2*k].imshow(X[i][b], cmap="gray", vmin=0, vmax=255); ax[1 + 2*k].axis("off")
        ax[1 + 2*k].set_title(f"slice {i} — raw (x{256/(b[0].stop-b[0].start):.1f})", fontsize=7)
        overlay(ax[2 + 2*k], X[i][b], Y[i][b], highlight=c, alpha=0.35)
        around = [NAMES[d] for d in np.unique(Y[i][b])[1:] if d != c]
        ax[2 + 2*k].set_title(f"↳ {int((Y[i]==c).sum())} px\n" + (", ".join(around[:3]) or "—"), fontsize=6)
    fig.suptitle(f"{c} — {NAMES[c]}", fontsize=11)
    plt.tight_layout(); plt.show()

show_organ(2)    # adrenal_gland_right: 37 px, the smallest structure in the set

In [ ]:
show_organ(48)   # spleen — changing the id is enough

---
## 5. Compare a model against the truth

Four panels: the scan, the (partial) truth, the prediction, and the agreement
map. Careful reading it — an organ predicted but absent from the truth is not
necessarily an error, it may be one of the 27 nobody outlined on that slice. The
agreement map therefore only judges the examined classes.

In [ ]:
import torch
from radium.infer import load_model, predict

CKPT = "checkpoints/ft145_256.pt"      # see CHECKPOINTS.md
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

model, hu = load_model(CKPT, DEVICE)
_, VAL = data.splits()
src = data.train_images_hu() if hu else X
PRED = predict(model, src[VAL], DEVICE, hu=hu)
SCORE = metric.dice_per_image(Y[VAL], PRED)
SCORE[~A[VAL]] = np.nan
print(f"{CKPT} — dice on the annotated classes: "
      f"{metric.dice_score_annotated(Y[VAL], PRED, A[VAL]):.4f}")

In [ ]:
def agreement(ax, scan, gt, pred, ann):
    """green = exact, red = wrong id, blue = missed, orange = spurious."""
    scored = np.isin(pred, np.flatnonzero(ann) + 1) | (gt > 0)
    rgba = np.zeros((*gt.shape, 4))
    rgba[(gt > 0) & (pred == gt)] = (0.15, 0.75, 0.25, 0.8)
    rgba[(gt > 0) & (pred != gt) & (pred > 0)] = (0.85, 0.15, 0.15, 0.8)
    rgba[(gt > 0) & (pred == 0)] = (0.20, 0.45, 0.95, 0.8)
    rgba[(gt == 0) & (pred > 0) & scored] = (1.00, 0.65, 0.10, 0.8)
    ax.imshow(scan, cmap="gray", vmin=0, vmax=255); ax.imshow(rgba, interpolation="nearest")
    ax.axis("off")

def compare(k):
    """k indexes the validation split, not the full set."""
    i = VAL[k]
    fig, ax = plt.subplots(1, 4, figsize=(13, 3.6))
    ax[0].imshow(X[i], cmap="gray", vmin=0, vmax=255); ax[0].axis("off")
    ax[0].set_title(f"slice {i} — dice {np.nanmean(SCORE[k]):.3f}", fontsize=8)
    overlay(ax[1], X[i], Y[VAL][k]); ax[1].set_title("truth (partial)", fontsize=8)
    overlay(ax[2], X[i], PRED[k]);   ax[2].set_title("prediction", fontsize=8)
    agreement(ax[3], X[i], Y[VAL][k], PRED[k], A[VAL][k])
    ax[3].set_title("green=exact red=wrong id\nblue=missed orange=spurious", fontsize=7)
    plt.tight_layout(); plt.show()

rank = [k for k in np.argsort(-np.nan_to_num(np.nanmean(SCORE, 1), nan=-1))
        if not np.isnan(np.nanmean(SCORE[k]))]
for k in [rank[0], rank[len(rank)//2], rank[-1]]:   # best, median, worst
    compare(k)

In [ ]:
# Where the model fails, class by class
per_class = np.nanmean(SCORE, axis=0)
order = np.argsort(np.nan_to_num(per_class, nan=-1))
print(f"{'id':>3} {'organ':<26} {'dice':>7} {'mean area':>10} {'slices/800':>11}")
for c0 in order[:12]:
    c = c0 + 1
    a = [(Y[i] == c).sum() for i in ANN if (Y[i] == c).any()]
    print(f"{c:>3} {NAMES[c]:<26} {per_class[c0]:>7.3f} "
          f"{(np.mean(a) if a else 0):>10.0f} {len(a):>11}")

---
## 6. The external set

Slices generated from TotalSegmentator, with **complete** supervision — every
organ outlined on every slice. Useful for checking by eye that the geometry and
the contrast imitate the challenge, which is what the transfer depends on.

In [ ]:
EXT = Path("external-data/external145")     # or external-data/external (1.20 mm/px)

if (EXT / "images.npy").exists():
    Xe = np.load(EXT / "images.npy", mmap_mode="r")
    Ye = np.load(EXT / "labels.npy", mmap_mode="r")
    print(f"{len(Xe)} external slices")

    rng = np.random.default_rng(0)
    sel = rng.choice(len(Xe), 5, replace=False)
    fig, ax = plt.subplots(2, 5, figsize=(15.5, 6.6))
    for k, i in enumerate(sel):
        img, lab = np.asarray(Xe[i]), np.asarray(Ye[i])
        ax[0, k].imshow(img, cmap="gray", vmin=0, vmax=255); ax[0, k].axis("off")
        ax[0, k].set_title(f"external {i}", fontsize=9)
        overlay(ax[1, k], img, lab)
        ax[1, k].set_title(f"{len(np.unique(lab)) - 1} organs", fontsize=8)
    plt.tight_layout(); plt.show()

    print(f"\nmean grey — challenge {X[:800].mean():.1f}  external {np.asarray(Xe[::37]).mean():.1f}")
    print(f"organs per slice — challenge {np.mean([len(np.unique(Y[i]))-1 for i in ANN]):.1f}  "
          f"external {np.mean([len(np.unique(np.asarray(Ye[i])))-1 for i in range(0, len(Ye), 97)]):.1f}")
else:
    print(f"{EXT} missing — see tools/make_external.py")

---
## 7. The 54 organs

Reference table: name, frequency, area, position. Rows carry laterality and
columns the back-to-front axis — the slices are rotated 90 degrees from the usual
radiological convention.

In [ ]:
rows = []
for c in range(1, 55):
    idx = [i for i in ANN if (Y[i] == c).any()]
    if not idx:
        rows.append((c, 0, 0, np.nan, np.nan)); continue
    ar = [(Y[i] == c).sum() for i in idx]
    rc = np.array([[np.nonzero(Y[i] == c)[0].mean() / 255,
                    np.nonzero(Y[i] == c)[1].mean() / 255] for i in idx])
    rows.append((c, len(idx), np.mean(ar), rc[:, 0].mean(), rc[:, 1].mean()))

print(f"{'id':>3} {'organ':<28} {'slices':>7} {'area':>7} {'side':>8} {'depth':>8}")
for c, n, a, r, k in sorted(rows, key=lambda t: -t[2]):
    side = "—" if np.isnan(r) else ("left" if r < 0.42 else "right" if r > 0.58 else "midline")
    depth = "—" if np.isnan(k) else ("back" if k < 0.42 else "front" if k > 0.58 else "middle")
    print(f"{c:>3} {NAMES[c]:<28} {n:>7} {a:>7.0f} {side:>8} {depth:>8}")